### INITIAL TURBULENCE-STATE STATISTICS AFTER MITgcm SPIN-UP
## Metrics:
###   - mesoscale EKE
###   - Re_eddy
###   - MLD from GLORYS
###   - bulk Richardson number


In [22]:
from pathlib import Path
import json
import numpy as np
import xarray as xr

In [23]:
# ============================================================
# SETTINGS
# ============================================================

NB_DIR = Path.cwd()

case_name = "run_domain_characterization2"          # edit
RUN_TITLE_INFO = "run_domain_characterization"     # edit

ds = xr.open_dataset(NB_DIR / f"../data/input/{case_name}.nc")
ds

<xarray.Dataset> Size: 884MB
Dimensions:  (T: 2, Z: 84, Y: 512, Xp1: 513, Yp1: 513, X: 512, Zl: 84)
Coordinates:
  * T        (T) float64 16B 0.0 800.0
    iter     (T) int32 8B ...
  * Z        (Z) float64 672B -1.0 -3.0 -5.0 ... -3.572e+03 -3.889e+03
  * Y        (Y) float64 4kB 250.0 750.0 1.25e+03 ... 2.552e+05 2.558e+05
  * Xp1      (Xp1) float64 4kB 0.0 500.0 1e+03 ... 2.55e+05 2.555e+05 2.56e+05
  * Yp1      (Yp1) float64 4kB 0.0 500.0 1e+03 ... 2.55e+05 2.555e+05 2.56e+05
  * X        (X) float64 4kB 250.0 750.0 1.25e+03 ... 2.552e+05 2.558e+05
  * Zl       (Zl) float64 672B 0.0 -2.0 -4.0 ... -3.419e+03 -3.725e+03
Data variables:
    U        (T, Z, Y, Xp1) float32 177MB ...
    V        (T, Z, Yp1, X) float32 177MB ...
    Temp     (T, Z, Y, X) float32 176MB ...
    S        (T, Z, Y, X) float32 176MB ...
    Eta      (T, Y, X) float32 2MB ...
    W        (T, Zl, Y, X) float32 176MB ...
Attributes: (12/18)
    MITgcm_version:  checkpoint69l
    build_user:      jgortemaker
    build_host:      cmp060
    build_date:      Fri May 22 09:34:05 CEST 2026
    MITgcm_URL:      http://mitgcm.org
    MITgcm_tag_id:   
    ...              ...
    nSy:             1
    nPx:             8
    nPy:             8
    Nx:              512
    Ny:              512
    Nr:              84

In [24]:
glorys_case = "GPGP_aug2020_3D_21.5_24.5_141.6_138.4"
GLORYS_MLD_FILE = (NB_DIR / f"../../OGCM/data/input/{glorys_case}.nc").resolve()
ds = xr.open_dataset(GLORYS_MLD_FILE)
ds

<xarray.Dataset> Size: 33MB
Dimensions:    (time: 31, depth: 46, latitude: 37, longitude: 39)
Coordinates:
  * time       (time) datetime64[ns] 248B 2020-08-01 2020-08-02 ... 2020-08-31
  * depth      (depth) float32 184B 0.494 1.541 2.646 ... 3.597e+03 3.992e+03
  * latitude   (latitude) float32 148B 21.5 21.58 21.67 ... 24.33 24.42 24.5
  * longitude  (longitude) float32 156B -141.6 -141.5 -141.4 ... -138.5 -138.4
Data variables:
    uo         (time, depth, latitude, longitude) float32 8MB ...
    vo         (time, depth, latitude, longitude) float32 8MB ...
    mlotst     (time, latitude, longitude) float32 179kB ...
    so         (time, depth, latitude, longitude) float32 8MB ...
    thetao     (time, depth, latitude, longitude) float32 8MB ...
    zos        (time, latitude, longitude) float32 179kB ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:        http://www.mercator-ocean.fr
    comment:           CMEMS product
    subset:source:     ARCO data downloaded from the Marine Data Store using ...
    subset:productId:  GLOBAL_MULTIYEAR_PHY_001_030
    subset:datasetId:  cmems_mod_glo_phy_my_0.083deg_P1D-m_202311
    subset:date:       2026-05-20T11:44:02.250Z

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

NB_DIR = Path.cwd()

case_name = "run_domain_characterization2"          # edit
glorys_case = "GPGP_aug2020_3D_21.5_24.5_141.6_138.4" # edit
RUN_TITLE_INFO = "run_domain_characterization"      # edit

# Optional GLORYS MLD file. If you already extracted a scalar MLD,
# set GLORYS_MLD_FILE = None and use MLD_VALUE_M below.
GLORYS_MLD_FILE = (NB_DIR / f"../../OGCM/data/input/{glorys_case}.nc").resolve()
GLORYS_MLD_VAR = "mlotst"
MITGCM_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()

# If no GLORYS MLD file is used, provide scalar MLD here [m]
# MLD_VALUE_M = 20.0

# Eddy-scale Reynolds number settings
L_EDDY_M = None       # if None, uses 0.25 * domain width
NU_EFF = 50.0         # [m2/s] effective horizontal viscosity

# Linear EOS constants
RHO0 = 1035.0         # [kg/m3]
ALPHA_T = 2.0e-4      # [1/K]
BETA_S = 7.4e-4       # [1/psu]
G = 9.81

# Minimum shear to avoid infinite Richardson numbers
MIN_DELTA_U2 = 1e-10

# Output
OUT_DIR = (NB_DIR / f"../results/{case_name}").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ATTRS_FILE = OUT_DIR / f"{case_name}_domain_characterization.json"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def center_u_to_tracer(u):
    """
    Center MITgcm U from Xp1 to X.
    Input shape : (Z, Y, Xp1)
    Output shape: (Z, Y, X)
    """
    return 0.5 * (u[:, :, :-1] + u[:, :, 1:])


def center_v_to_tracer(v):
    """
    Center MITgcm V from Yp1 to Y.
    Input shape : (Z, Yp1, X)
    Output shape: (Z, Y, X)
    """
    return 0.5 * (v[:, :-1, :] + v[:, 1:, :])


def crop_to_common(*arrays):
    """
    Crop arrays to common trailing horizontal shape.
    Assumes the final two dimensions are Y, X.
    """
    ny = min(a.shape[-2] for a in arrays)
    nx = min(a.shape[-1] for a in arrays)
    return tuple(a[..., :ny, :nx] for a in arrays)


def positive_down_depth_from_Z(ds):
    """
    MITgcm Z is usually negative downward.
    Returns positive-down depth vector [m].
    """
    z = np.asarray(ds["Z"].values, dtype=float)

    if np.nanmean(z) < 0:
        z = -z

    return z


def interp_3d_to_depth(field_zyx, z_m, target_depth_m):
    """
    Interpolate a 3D field with shape (Z, Y, X) to a scalar depth.
    z_m must be positive downward.
    Returns field at target depth with shape (Y, X).
    """
    field_zyx = np.asarray(field_zyx, dtype=float)
    z_m = np.asarray(z_m, dtype=float)

    if target_depth_m <= z_m[0]:
        return field_zyx[0, :, :]

    if target_depth_m >= z_m[-1]:
        return field_zyx[-1, :, :]

    k2 = int(np.searchsorted(z_m, target_depth_m))
    k1 = k2 - 1

    z1 = z_m[k1]
    z2 = z_m[k2]

    w = (target_depth_m - z1) / (z2 - z1)

    return (1.0 - w) * field_zyx[k1, :, :] + w * field_zyx[k2, :, :]


def density_linear(temp, salt):
    """
    Linear EOS used only for density differences.
    Reference T/S constants cancel in Delta rho, so they are omitted.
    """
    return RHO0 * (1.0 - ALPHA_T * temp + BETA_S * salt)


def get_time_value(ds, dim_name, index):
    """
    Return raw coordinate value and printable string for a time coordinate.
    """
    value = ds[dim_name].isel({dim_name: index}).values

    if np.issubdtype(np.asarray(value).dtype, np.datetime64):
        value_str = np.datetime_as_string(value, unit="D")
        value_raw = value_str
    else:
        value_raw = float(np.asarray(value))
        value_str = f"{value_raw:g}"

    return value_raw, value_str


# ============================================================
# OPEN DATA
# ============================================================

print(f"Opening MITgcm file:\n{MITGCM_FILE}")
ds = xr.open_dataset(MITGCM_FILE)

print(f"\nOpening GLORYS file:\n{GLORYS_FILE}")
ds_glorys = xr.open_dataset(GLORYS_FILE)

# Print selected MITgcm time
mitgcm_T_raw, mitgcm_T_str = get_time_value(ds, "T", MITGCM_TIME_INDEX)

print("\nSelected MITgcm post-spinup snapshot:")
print(f"  selected index : {MITGCM_TIME_INDEX}")
print(f"  T[{MITGCM_TIME_INDEX}]         : {mitgcm_T_str}")
print(f"  T units        : {ds['T'].attrs.get('units', 'not specified')}")

# Print selected GLORYS time
glorys_time_raw, glorys_time_str = get_time_value(ds_glorys, "time", GLORYS_TIME_INDEX)

print("\nSelected GLORYS MLD snapshot:")
print(f"  selected index : {GLORYS_TIME_INDEX}")
print(f"  time[{GLORYS_TIME_INDEX}]      : {glorys_time_str}")


# ============================================================
# EXTRACT MITgcm FIELDS AT FIRST POST-SPINUP TIME
# ============================================================

u_raw = ds["U"].isel(T=MITGCM_TIME_INDEX).values        # (Z, Y, Xp1)
v_raw = ds["V"].isel(T=MITGCM_TIME_INDEX).values        # (Z, Yp1, X)
temp = ds["Temp"].isel(T=MITGCM_TIME_INDEX).values      # (Z, Y, X)
salt = ds["S"].isel(T=MITGCM_TIME_INDEX).values         # (Z, Y, X)

u = center_u_to_tracer(u_raw)                           # (Z, Y, X)
v = center_v_to_tracer(v_raw)                           # (Z, Y, X)

u, v, temp, salt = crop_to_common(u, v, temp, salt)

z_m = positive_down_depth_from_Z(ds)

print("\nMITgcm field shapes after centering/cropping:")
print(f"  U centered : {u.shape}")
print(f"  V centered : {v.shape}")
print(f"  Temp       : {temp.shape}")
print(f"  S          : {salt.shape}")
print(f"  Z levels   : {len(z_m)}")
print(f"  depth range: {np.nanmin(z_m):.2f} to {np.nanmax(z_m):.2f} m")


# ============================================================
# 1. POST-SPINUP SURFACE EKE
# ============================================================

u_surf = u[0, :, :]
v_surf = v[0, :, :]

u_prime = u_surf - np.nanmean(u_surf)
v_prime = v_surf - np.nanmean(v_surf)

eke_field = 0.5 * (u_prime**2 + v_prime**2)

eke_mean = float(np.nanmean(eke_field))
eke_median = float(np.nanmedian(eke_field))
eke_std = float(np.nanstd(eke_field))

u_eddy = float(np.sqrt(2.0 * eke_mean))

print("\n1. Post-spinup surface EKE:")
print(f"  EKE mean          : {eke_mean:.6e} m2/s2")
print(f"  EKE median        : {eke_median:.6e} m2/s2")
print(f"  EKE std           : {eke_std:.6e} m2/s2")
print(f"  U_eddy=sqrt(2EKE) : {u_eddy:.6e} m/s")


# ============================================================
# 2. EFFECTIVE EDDY-SCALE REYNOLDS NUMBER
# ============================================================

x = np.asarray(ds["X"].values, dtype=float)
domain_width_m = float(np.nanmax(x) - np.nanmin(x))

if L_EDDY_M is None:
    L_eddy_m = 0.25 * domain_width_m
else:
    L_eddy_m = float(L_EDDY_M)

re_eddy = float(u_eddy * L_eddy_m / NU_EFF)

print("\n2. Effective eddy-scale Reynolds number:")
print(f"  domain width      : {domain_width_m:.2f} m")
print(f"  L_eddy            : {L_eddy_m:.2f} m")
print(f"  nu_eff            : {NU_EFF:.6e} m2/s")
print(f"  Re_eddy           : {re_eddy:.6e}")


# ============================================================
# 3. MLD FROM GLORYS
# ============================================================

if USE_GLORYS_MLD:
    mld_field = ds_glorys["mlotst"].isel(time=GLORYS_TIME_INDEX).values
    mld_vals = mld_field[np.isfinite(mld_field)]

    mld_mean = float(np.nanmean(mld_vals))
    mld_median = float(np.nanmedian(mld_vals))
    mld_std = float(np.nanstd(mld_vals))

    mld_source = str(GLORYS_FILE)
    mld_for_ri = mld_median

else:
    mld_mean = float(MLD_VALUE_M)
    mld_median = float(MLD_VALUE_M)
    mld_std = np.nan

    mld_source = "scalar_user_input"
    mld_for_ri = float(MLD_VALUE_M)

print("\n3. Mixed-layer depth:")
print(f"  MLD source        : {mld_source}")
print(f"  MLD mean          : {mld_mean:.2f} m")
print(f"  MLD median used   : {mld_for_ri:.2f} m")
print(f"  MLD std           : {mld_std:.2f} m")


# ============================================================
# 4. BULK RICHARDSON NUMBER OVER MLD
# ============================================================

rho = density_linear(temp, salt)

rho_surf = rho[0, :, :]
u_surf = u[0, :, :]
v_surf = v[0, :, :]

rho_mld = interp_3d_to_depth(rho, z_m, mld_for_ri)
u_mld = interp_3d_to_depth(u, z_m, mld_for_ri)
v_mld = interp_3d_to_depth(v, z_m, mld_for_ri)

rho_surf, rho_mld, u_surf, v_surf, u_mld, v_mld = crop_to_common(
    rho_surf, rho_mld, u_surf, v_surf, u_mld, v_mld
)

delta_rho = rho_mld - rho_surf

delta_u2 = (
    (u_surf - u_mld) ** 2
    + (v_surf - v_mld) ** 2
)

delta_u2_safe = np.where(delta_u2 < MIN_DELTA_U2, np.nan, delta_u2)

rib_field = G * delta_rho * mld_for_ri / (RHO0 * delta_u2_safe)

rib_median = float(np.nanmedian(rib_field))
rib_mean = float(np.nanmean(rib_field))
rib_p25 = float(np.nanpercentile(rib_field, 25))
rib_p75 = float(np.nanpercentile(rib_field, 75))

print("\n4. Bulk Richardson number:")
print(f"  Ri_b median       : {rib_median:.6e}")
print(f"  Ri_b mean         : {rib_mean:.6e}")
print(f"  Ri_b p25          : {rib_p25:.6e}")
print(f"  Ri_b p75          : {rib_p75:.6e}")


# ============================================================
# STORE ATTRS JSON
# ============================================================

attrs = {
    "case_name": case_name,

    "mitgcm_file": str(MITGCM_FILE),
    "glorys_file": str(GLORYS_FILE),

    "mitgcm_time_index_used": int(MITGCM_TIME_INDEX),
    "mitgcm_T_value_used": mitgcm_T_raw,
    "mitgcm_T_units": ds["T"].attrs.get("units", "not specified"),

    "glorys_time_index_used": int(GLORYS_TIME_INDEX),
    "glorys_time_value_used": glorys_time_raw,

    "statistics_description": {
        "EKE": (
            "Domain-mean post-spinup surface EKE after removing the spatial "
            "domain-mean surface velocity. This includes all resolved MITgcm "
            "scales unless the input field has been filtered."
        ),
        "Re_eddy": (
            "Effective eddy-scale Reynolds number using "
            "U_eddy=sqrt(2*EKE), L_eddy, and nu_eff."
        ),
        "MLD": (
            "Domain mean/median/std of GLORYS mlotst. Median MLD is used "
            "as the vertical scale for Ri_b."
        ),
        "Ri_b": (
            "Bulk Richardson number between the surface and GLORYS median MLD, "
            "using post-spinup MITgcm Temp/S/U/V."
        ),
    },

    "eke_mean_m2_s2": eke_mean,
    "eke_median_m2_s2": eke_median,
    "eke_std_m2_s2": eke_std,
    "u_eddy_m_s": u_eddy,

    "domain_width_m": domain_width_m,
    "L_eddy_m": float(L_eddy_m),
    "nu_eff_m2_s": float(NU_EFF),
    "Re_eddy": re_eddy,

    "mld_mean_m": mld_mean,
    "mld_median_m": mld_median,
    "mld_std_m": None if not np.isfinite(mld_std) else float(mld_std),
    "mld_source": mld_source,

    "Ri_b_median": rib_median,
    "Ri_b_mean": rib_mean,
    "Ri_b_p25": rib_p25,
    "Ri_b_p75": rib_p75,

    "rho0_kg_m3": RHO0,
    "alpha_T_1_K": ALPHA_T,
    "beta_S_1_psu": BETA_S,
    "g_m_s2": G,
}

with open(OUT_ATTRS_FILE, "w") as f:
    json.dump(attrs, f, indent=4)

print(f"\nSaved attrs file:\n{OUT_ATTRS_FILE}")

Opening MITgcm file:
C:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\data\input\run_domain_characterization2.nc

Opening GLORYS file:
C:\Users\Jelle Gortemaker\Documents\Thesis\OGCM\data\input\GPGP_aug2020_3D_21.5_24.5_141.6_138.4.nc

Selected MITgcm post-spinup snapshot:
  selected index : 0
  T[0]         : 0
  T units        : s

Selected GLORYS MLD snapshot:
  selected index : 0
  time[0]      : 2020-08-01

MITgcm field shapes after centering/cropping:
  U centered : (84, 512, 512)
  V centered : (84, 512, 512)
  Temp       : (84, 512, 512)
  S          : (84, 512, 512)
  Z levels   : 84
  depth range: 1.00 to 3889.25 m

1. Post-spinup surface EKE:
  EKE mean          : 1.095136e-02 m2/s2
  EKE median        : 7.768351e-03 m2/s2
  EKE std           : 1.075961e-02 m2/s2
  U_eddy=sqrt(2EKE) : 1.479956e-01 m/s

2. Effective eddy-scale Reynolds number:
  domain width      : 255500.00 m
  L_eddy            : 63875.00 m
  nu_eff            : 5.000000e+01 m2/s
  Re_eddy  